# 使用 Codex 构建迭代式修复循环

本 cookbook 讨论的是闭环智能体工作流：智能体生成输出、对其进行验证，并利用反馈改进下一轮。

我们将探索一个文档可靠性工作流，用于检测、修复和验证过时或已失效的 API 与 SDK 示例。示例中使用的是从本 Cookbook 仓库改编而来、有意设置为过时状态的 notebook。

我们将使用 Codex 构建这个智能体循环。Codex 会审查当前状态、进行有针对性的修改、运行验证，并在反馈显示仍有问题时重复这一过程。

notebook 任务只是一个示例。只要智能体的输出可以通过可信的反馈来衡量，这种模式就同样适用。

该工作流包含三个阶段：

- **审查：** 检查当前产物并返回结构化的发现，不编辑任何文件。
- **修复：** 根据审查发现和最新的验证反馈，对复制出的产物进行有针对性的编辑。
- **验证：** 运行相关检查，并报告仍需处理的问题。

验证让循环得以闭合。修复后的 notebook 必须通过那些真正重要的检查，任何剩余问题都会成为下一轮修复的输入。


<p align="center">
  <img src="../../images/codex_iterative_repair_loop.png" alt="Codex iterative repair loop for technical documentation" width="350"/>
</p>


## 设置

本 notebook 以无头（headless）模式使用 [Codex CLI](https://developers.openai.com/codex/cli)，因此修复步骤可以在 Python 单元中运行，而不必通过聊天界面。第一个代码单元会安装该 CLI；如果你已经安装，可以跳过该单元。

在运行实时修复循环之前，请在环境中设置 `OPENAI_API_KEY`。

本 notebook 默认使用一个速度较快的修复模型，以便完整示例能在合理的时间内完成。若要尝试其他模型，请在开始前设置 `REPAIR_MODEL`。安装单元固定了一个已知的 Codex CLI 版本以保证可复现性；当你需要更新的 CLI 行为时，请有意识地更新该版本。


In [ ]:
!npm install -g @openai/codex@0.130.0

In [2]:
import concurrent.futures
import json
import os
import shlex
import shutil
import subprocess
import tempfile
from pathlib import Path
from typing import Any

CANDIDATE_EXAMPLE_DIRS = [Path("."), Path("examples/codex")]
EXAMPLE_DIR = next((base for base in CANDIDATE_EXAMPLE_DIRS if (base / "data" / "docs").exists()), None)

if EXAMPLE_DIR is None:
    raise RuntimeError(
        "This notebook needs its companion sample notebooks. "
        "Download the data folder that ships with this example and place it next to "
        "this notebook as ./data/docs, or run from a checkout where examples/codex/data/docs exists."
    )

DATA_DIR = EXAMPLE_DIR / "data" / "docs"
DEFAULT_RUNS_DIR = Path(tempfile.gettempdir()) / "codex_iterative_repair_loop_outputs"
RUNS_DIR = Path(os.getenv("CODEX_REPAIR_RUNS_DIR", str(DEFAULT_RUNS_DIR))).expanduser()
RUNS_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
MODEL = os.getenv("REPAIR_MODEL", "gpt-5.4-mini")
COOKBOOK_CHAT_MODEL = os.getenv("COOKBOOK_CHAT_MODEL", "gpt-5.5")
REPAIR_REASONING_EFFORT = os.getenv("REPAIR_REASONING_EFFORT", "low")

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Set the OPENAI_API_KEY environment variable before running the live Codex repair loop.")

CODEX_CLI = shutil.which("codex")
if CODEX_CLI is None:
    raise RuntimeError("Run the install cell before continuing; Codex CLI is not on PATH.")


## 加载示例产物

下面的单元会加载三个配套 notebook，并汇总驱动修复循环的元数据。

这些示例刻意保持很小。它们运行很快，但仍能完整演练这一架构：审查发现实质性问题，修复进行有针对性的编辑，验证为下一轮生成反馈。

如果你只单独下载了本 notebook，请同时下载配套的 `data/docs/` 文件夹，并在运行下面的单元之前将其放在 notebook 旁边。代码要求这些示例 notebook 在本地可用。

在本示例中，验证会端到端地执行每个修复后的 notebook。在其他领域，验证可能是单元测试、策略检查、schema 校验器、仿真或人工审批步骤。关键在于，失败会变成结构化的反馈，而不是死胡同。


In [4]:
NOTEBOOKS = [
    DATA_DIR / "qdrant_embeddings_search_pre_repair.ipynb",
    DATA_DIR / "getting_started_evals_pre_repair.ipynb",
    DATA_DIR / "knowledge_retrieval_pre_repair.ipynb",
]


def read_notebook(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def case_metadata(path: Path) -> dict[str, Any]:
    return read_notebook(path).get("metadata", {}).get("codex_case_study", {})


cases = []
for notebook_path in NOTEBOOKS:
    notebook = read_notebook(notebook_path)
    metadata = notebook.get("metadata", {}).get("codex_case_study", {})
    repair_story = metadata.get("repair_story", {})
    cases.append(
        {
            "notebook": notebook_path.name,
            "cells": len(notebook["cells"]),
            "code_cells": sum(cell["cell_type"] == "code" for cell in notebook["cells"]),
            "source": metadata.get("source_path"),
            "target_iteration": repair_story.get("target_iteration"),
            "repair_depth": repair_story.get("repair_depth", ""),
        }
    )

cases


[{'notebook': 'qdrant_embeddings_search_pre_repair.ipynb',
  'cells': 5,
  'code_cells': 4,
  'source': 'examples/vector_databases/qdrant/Using_Qdrant_for_embeddings_search.ipynb',
  'target_iteration': 1,
  'repair_depth': 'One-pass cleanup: modernize the local Qdrant query path and clarify the sampled fixture framing.'},
 {'notebook': 'getting_started_evals_pre_repair.ipynb',
  'cells': 5,
  'code_cells': 4,
  'source': 'examples/evaluation/Getting_Started_with_OpenAI_Evals.ipynb',
  'target_iteration': 2,
  'repair_depth': 'Two-pass cleanup: first modernize the obvious stale Evals flow, then use validation feedback to remove result-log brittleness.'},
 {'notebook': 'knowledge_retrieval_pre_repair.ipynb',
  'cells': 5,
  'code_cells': 4,
  'source': 'examples/How_to_call_functions_for_knowledge_retrieval.ipynb',
  'target_iteration': 3,
  'repair_depth': 'Three-pass cleanup: modernize model/API shape, then tighten runnable local setup, then restore the full retrieval teaching flow.'}

## 定义业务规则和问题分类

在让 Codex 审查或修复产物之前，先给它一份简短的共享约定。这能让循环聚焦于真正重要的问题，而不是让模型从零开始推断每一条产品和风格规则。

下面的规则定义了这些示例 notebook 的“好”是什么意思：使用当前的 API 模式、设置说明清晰、本地示例可运行，并保留原有的教学目标。在其他工作流中，这份约定会描述该领域的事实来源。


In [ ]:
business_rules = {
    "preferred_chat_model": COOKBOOK_CHAT_MODEL,
    "preferred_embedding_model": "text-embedding-3-large",
    "modernize": [
        "client.chat.completions.create -> client.responses.create",
        "legacy function-calling schemas -> current tools schema",
        "qdrant.search -> qdrant.query_points",
        "oaieval CLI examples -> current Evals API workflow",
    ],
    "reader_experience": [
        "Make fresh-environment setup explicit.",
        "Keep the included examples runnable with local data and the standard library.",
        "Keep sample repairs self-contained unless the notebook explicitly teaches external setup.",
        "Remove manual result-file placeholders.",
        "State runtime prerequisites and side effects before readers run cells.",
        "Preserve the original teaching goal while modernizing the implementation.",
    ],
}

business_rules


## 定义结构化输出

每个阶段都返回结构化数据，让下一个阶段有具体的内容可用。

审查返回发现的问题。修复返回变更摘要以及更新后产物的路径。验证返回留给下一轮的剩余差异（delta）。借助结构化的交接，循环更易于调试、重新运行，并适配其他类型的产物。


In [6]:
def object_schema(properties: dict[str, Any], required: list[str] | None = None) -> dict[str, Any]:
    return {
        "type": "object",
        "properties": properties,
        "required": required or list(properties),
        "additionalProperties": False,
    }


def string_array() -> dict[str, Any]:
    return {"type": "array", "items": {"type": "string"}}


finding_schema = object_schema(
    {
        "artifact": {"type": "string"},
        "issue_type": {"type": "string"},
        "severity": {"type": "string"},
        "description": {"type": "string"},
        "suggested_fix_direction": {"type": "string"},
    }
)

review_schema = object_schema(
    {"findings": {"type": "array", "items": finding_schema}}
)

fix_schema = object_schema(
    {
        "artifact": {"type": "string"},
        "iteration": {"type": "integer"},
        "changes_made": string_array(),
        "unresolved_items": string_array(),
        "updated_artifact_path": {"type": "string"},
    }
)

validation_case_schema = object_schema(
    {
        "name": {"type": "string"},
        "passed": {"type": "boolean"},
        "severity": {"type": "string"},
        "evidence": {"type": "string"},
        "feedback": {"type": "string"},
    }
)

validation_schema = object_schema(
    {
        "overall_passed": {"type": "boolean"},
        "cases": {"type": "array", "items": validation_case_schema},
        "remaining_delta": string_array(),
    }
)

## 审查阶段

审查阶段读取产物并返回结构化的发现。它不运行验证，也不编辑文件。这种分离让第一步保持聚焦：在做任何改动之前，先识别可能存在的问题。

我们将审查提示词连同一个 JSON schema 发送给 `codex exec`。schema 让结果保持机器可读，这样后面的单元就可以把发现直接传入修复提示词，而不必从上一次回答的文字中抓取信息。


In [7]:
def notebook_text(path: Path, max_chars: int = 7000) -> str:
    chunks = []
    for index, cell in enumerate(read_notebook(path)["cells"]):
        source = "".join(cell.get("source", []))
        chunks.append(f"cell {index} ({cell['cell_type']})\n{source}")
    text = "\n\n".join(chunks)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n\n[truncated for prompt size]"


def run_command(command: str, *, stdin: str | None = None, cwd: Path | None = None, timeout: int | None = None):
    cwd = Path.cwd() if cwd is None else cwd
    return subprocess.run(
        shlex.split(command),
        input=stdin,
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=timeout,
        check=False,
    )


def run_codex_json(prompt: str, schema: dict[str, Any], run_dir: Path) -> dict[str, Any]:
    run_dir.mkdir(parents=True, exist_ok=True)
    prompt_file = run_dir / "prompt.txt"
    schema_file = run_dir / "schema.json"
    answer_file = run_dir / "answer.json"

    prompt_file.write_text(prompt, encoding="utf-8")
    schema_file.write_text(json.dumps(schema, indent=2), encoding="utf-8")

    command = f"""
    {CODEX_CLI} exec
      --model {MODEL}
      --sandbox workspace-write
      --ask-for-approval never
      --config model_reasoning_effort={REPAIR_REASONING_EFFORT}
      --output-schema {schema_file}
      --output-last-message {answer_file}
      -
    """
    result = run_command(command, stdin=prompt)
    (run_dir / "stdout.txt").write_text(result.stdout, encoding="utf-8")
    (run_dir / "stderr.txt").write_text(result.stderr, encoding="utf-8")

    if result.returncode != 0:
        raise RuntimeError(f"Codex exited with {result.returncode}. See {run_dir / 'stderr.txt'}.")

    return json.loads(answer_file.read_text(encoding="utf-8"))


def review_notebook(path: Path, run_dir: Path) -> list[dict[str, Any]]:
    prompt = "\n".join(
        [
            "You are reviewing a public OpenAI Cookbook notebook before publication.",
            f"Artifact: {path.name}",
            "Find issues that would make the notebook stale, hard to run, or confusing for a developer reader.",
            "Do not execute the notebook or edit files.",
            "Use concise issue_type labels such as stale_model, deprecated_api, setup_gap, runtime_risk, or clarity_issue.",
            f"Business rules: {json.dumps(business_rules)}",
            "Base findings only on the notebook content below.",
            "Keep the findings focused; three strong findings are better than a long list.",
            "",
            notebook_text(path),
        ]
    )
    return run_codex_json(prompt, review_schema, run_dir)["findings"]


In [ ]:
def run_initial_review(path: Path) -> tuple[str, list[dict[str, Any]]]:
    return path.name, review_notebook(path, RUNS_DIR / "initial_review" / path.stem)


with concurrent.futures.ThreadPoolExecutor(max_workers=min(3, len(NOTEBOOKS))) as executor:
    initial_reviews = dict(executor.map(run_initial_review, NOTEBOOKS))

initial_reviews


## 修复阶段

修复阶段会获得当前产物、审查发现、业务规则，以及上一轮的所有验证反馈。随着循环不断积累信息，提示词也会变得更加具体。

Codex 在迭代目录中编辑一份副本，并返回一段简短的变更摘要。循环不会假定编辑一定有效；这一点由下一步的验证来判定。


In [9]:
def repair_prompt(path: Path, updated_path: Path, findings: list[dict[str, Any]], remaining_delta: list[str], iteration: int) -> str:
    repair_story = case_metadata(path).get("repair_story", {})
    return "\n".join(
        [
            "You are repairing a copy of a public OpenAI Cookbook notebook.",
            f"Source notebook: {path}",
            f"Editable copy: {updated_path}",
            f"Iteration: {iteration}",
            "Make the smallest useful edits that address the review findings and validation delta.",
            "Preserve the notebook's teaching flow and original purpose.",
            "Keep sample repairs self-contained unless the notebook explicitly teaches external setup.",
            "For staged examples, focus on the most important remaining issue for this pass instead of rewriting everything at once.",
            "Edit only the editable copy. Do not claim the notebook passes validation.",
            f"Repair depth: {json.dumps(repair_story, indent=2)}",
            f"Business rules: {json.dumps(business_rules, indent=2)}",
            f"Review findings: {json.dumps(findings, indent=2)}",
            f"Remaining validation delta: {json.dumps(remaining_delta, indent=2)}",
        ]
    )


def repair_notebook(path: Path, iteration: int, findings: list[dict[str, Any]], remaining_delta: list[str], case_dir: Path) -> dict[str, Any]:
    updated_path = case_dir / "updated.ipynb"
    updated_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, updated_path)

    prompt = repair_prompt(path, updated_path, findings, remaining_delta, iteration)
    return run_codex_json(prompt, fix_schema, case_dir / "repair")


## 验证阶段

验证就像一个小型评估（eval）。我们定义期望的行为，运行相关检查，并让评判模型根据该评分标准为结果打分。

对于文档示例，首先要做的是执行。许多 notebook 问题只有在运行时才会暴露：缺少导入、文件路径过时、某个单元依赖旧的 API 响应，或者设置说明对作者来说很清楚，但对初次阅读的读者并不清楚。

如果验证失败，失败本身就会成为下一轮修复的依据。这让下一次修复建立在观察到的行为之上，而不只是基于 diff 中看起来正确的内容。


In [10]:
VALIDATION_CASES = [
    {
        "name": "api_modernization",
        "question": "Does the notebook avoid stale OpenAI API patterns, legacy function-calling syntax, and outdated model names?",
    },
    {
        "name": "setup_reproducibility",
        "question": "Could a reader run the notebook from a fresh environment without hidden manual steps?",
    },
    {
        "name": "artifact_integrity",
        "question": "Did the update preserve the notebook's teaching flow and avoid deleting substantive cells?",
    },
]


def short_output(value: Any, limit: int = 1200) -> str:
    if value is None:
        return ""
    if isinstance(value, bytes):
        value = value.decode("utf-8", errors="replace")
    return str(value)[-limit:]


def execute_notebook(path: Path) -> dict[str, Any]:
    code_cells = sum(cell["cell_type"] == "code" for cell in read_notebook(path)["cells"])
    command = f"jupyter nbconvert --to notebook --execute --inplace {path.name}"

    try:
        result = run_command(
            command,
            cwd=path.parent,
            timeout=int(os.getenv("SAMPLE_NOTEBOOK_TIMEOUT_SECONDS", "300")),
        )
    except FileNotFoundError:
        return {
            "status": "failed",
            "executed_code_cells": 0,
            "error": "Jupyter or nbconvert is not installed or is not available on PATH.",
            "summary": "Install Jupyter with nbconvert before running the validation loop.",
        }
    except subprocess.TimeoutExpired as exc:
        return {
            "status": "failed",
            "executed_code_cells": 0,
            "error": f"Notebook execution timed out after {exc.timeout} seconds.",
            "summary": short_output(exc.stderr or exc.stdout),
        }

    output = result.stderr or result.stdout
    return {
        "status": "passed" if result.returncode == 0 else "failed",
        "executed_code_cells": code_cells if result.returncode == 0 else 0,
        "error": "" if result.returncode == 0 else f"Notebook execution exited with code {result.returncode}.",
        "summary": short_output(output),
    }

def validation_prompt(updated_path: Path, before_path: Path, execution: dict[str, Any], iteration: int) -> str:
    repair_story = case_metadata(before_path).get("repair_story", {})
    return "\n".join(
        [
            "You are judging a repaired OpenAI Cookbook notebook.",
            f"Iteration: {iteration}",
            "Score each validation case independently and give concise feedback for the next repair pass.",
            "Set overall_passed to false when execution failed or any case has a material issue.",
            "When execution failed, include the failure in remaining_delta so the next repair pass can address it.",
            "Use the business rules as the source of truth for current model names and API targets.",
            "Do not mark the preferred embedding model or preferred chat model as stale.",
            "For local examples, do not require extra services or package installs when the notebook says it is intentionally self-contained.",
            f"Repair depth: {json.dumps(repair_story, indent=2)}",
            f"Business rules: {json.dumps(business_rules, indent=2)}",
            f"Validation cases: {json.dumps(VALIDATION_CASES, indent=2)}",
            f"Execution evidence: {json.dumps(execution, indent=2)}",
            f"Original cell count: {len(read_notebook(before_path)['cells'])}",
            f"Updated cell count: {len(read_notebook(updated_path)['cells'])}",
            "",
            notebook_text(updated_path),
        ]
    )


def staged_delta(before_path: Path, iteration: int) -> list[str]:
    repair_story = case_metadata(before_path).get("repair_story", {})
    target = int(repair_story.get("target_iteration") or 1)
    if iteration >= target:
        return []
    depth = repair_story.get("repair_depth", "This case is intentionally staged across multiple repair passes.")
    return [f"Continue to iteration {iteration + 1}: {depth}"]


def evaluate_notebook(updated_path: Path, before_path: Path, run_dir: Path, iteration: int) -> dict[str, Any]:
    execution = execute_notebook(updated_path)
    judged = run_codex_json(validation_prompt(updated_path, before_path, execution, iteration), validation_schema, run_dir)
    failed_cases = [case for case in judged["cases"] if not case["passed"]]
    execution_delta = []
    if execution["status"] != "passed":
        execution_delta.append(f"Execution failed: {execution.get('error') or execution.get('summary')}")

    stage_delta = staged_delta(before_path, iteration)
    return {
        "passed": judged["overall_passed"] and execution["status"] == "passed" and not stage_delta,
        "execution_status": execution["status"],
        "executed_code_cells": execution["executed_code_cells"],
        "execution_summary": execution["summary"],
        "findings": failed_cases,
        "remaining_delta": execution_delta + stage_delta + judged["remaining_delta"],
    }


## 保存每次迭代的输出

每次迭代都会写入一个 `record.json` 文件，在本示例中还会在 `CODEX_REPAIR_RUNS_DIR/iteration_N/<sample_name>/` 下写入修复后的 notebook。如果你没有设置 `CODEX_REPAIR_RUNS_DIR`，notebook 会写入系统临时目录，从而保持正常的仓库检出目录干净。

这些文件就是审计轨迹。你可以看到审查发现了什么、Codex 改了什么、执行是否通过，以及哪些反馈被带入了下一次迭代。

`record.json` 文件是一次循环尝试的凭据。它把各阶段之间的交接内容集中保存在一个地方：

```json
{
  "review": [{"issue_type": "deprecated_api", "severity": "high"}],
  "repair": {
    "changes_made": ["Updated the notebook to use the current API pattern."],
    "updated_artifact_path": "/tmp/codex_iterative_repair_loop_outputs/iteration_1/sample/updated.ipynb"
  },
  "validation": {
    "passed": false,
    "remaining_delta": ["One setup instruction is still unclear."]
  }
}
```

正是这种紧凑的记录，让维护者无需从 notebook diff 和终端日志中重建整个运行过程，就能审查这个循环。


In [11]:
def save_json(payload: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")


def iteration_dir(number: int) -> Path:
    path = RUNS_DIR / f"iteration_{number}"
    path.mkdir(parents=True, exist_ok=True)
    return path


## 运行第 1 次迭代

每个 notebook 案例彼此独立，因此我们并发处理这些案例。这让演示保持快速，同时每个示例仍然经历相同的审查、修复和验证流程。

第 1 次迭代复用前面审查单元得到的初始审查发现。这一轮结束后，检查返回的布尔值：通过的案例可以停止，失败的案例会把验证反馈带入下一轮。


In [12]:
current_notebooks = {path.name: path for path in NOTEBOOKS}
history: dict[int, dict[str, Any]] = {}


def review_findings_for(original: Path, current_path: Path, case_dir: Path, previous_results: dict[str, Any] | None) -> list[dict[str, Any]]:
    if previous_results is None:
        return initial_reviews[original.name]
    return review_notebook(current_path, case_dir / "review")


def run_case(number: int, original: Path, run_dir: Path, previous_results: dict[str, Any] | None) -> tuple[str, dict[str, Any], Path]:
    name = original.name
    case_dir = run_dir / original.stem
    current_path = current_notebooks[name]

    findings = review_findings_for(original, current_path, case_dir, previous_results)
    delta = [] if previous_results is None else previous_results[name]["validation"]["remaining_delta"]
    repair = repair_notebook(current_path, number, findings, delta, case_dir)
    updated_path = Path(repair["updated_artifact_path"])
    validation = evaluate_notebook(updated_path, current_path, case_dir / "evaluation", number)

    record = {"review": findings, "repair": repair, "validation": validation}
    save_json(record, case_dir / "record.json")
    return name, record, updated_path


def run_iteration(number: int, previous_results: dict[str, Any] | None = None) -> dict[str, Any]:
    results = {}
    updates = {}
    run_dir = iteration_dir(number)

    with concurrent.futures.ThreadPoolExecutor(max_workers=min(3, len(NOTEBOOKS))) as executor:
        futures = [executor.submit(run_case, number, original, run_dir, previous_results) for original in NOTEBOOKS]
        for future in concurrent.futures.as_completed(futures):
            name, record, updated_path = future.result()
            results[name] = record
            updates[name] = updated_path

    current_notebooks.update(updates)
    history[number] = results
    return results


iteration_1 = run_iteration(1)
{name: result["validation"]["passed"] for name, result in iteration_1.items()}


{'qdrant_embeddings_search_pre_repair.ipynb': True,
 'getting_started_evals_pre_repair.ipynb': False,
 'knowledge_retrieval_pre_repair.ipynb': False}

## 运行第 2 次迭代

从第 2 次迭代开始，循环的价值开始显现。Codex 不再只依据最初的审查进行工作；它还能看到验证过程中发生了什么。

这改变了任务的性质。我们不再要求大范围重写，而是根据上一轮运行的证据——哪些执行了、哪些通过了、哪些仍需处理——请求下一步有用的修复。

对于随附的分阶段测试用例，这一轮的设计目标是解决中等难度的 Evals 案例，而难度更高的 Knowledge Retrieval 案例则会带着更小、更具体的差异继续进行。


In [13]:
iteration_2 = run_iteration(2, iteration_1)
{name: result["validation"]["passed"] for name, result in iteration_2.items()}


{'getting_started_evals_pre_repair.ipynb': True,
 'qdrant_embeddings_search_pre_repair.ipynb': True,
 'knowledge_retrieval_pre_repair.ipynb': False}

## 运行第 3 次迭代

第 3 次迭代聚焦于难度最高的文档案例。

Knowledge Retrieval 测试用例需要将 API 形态现代化，同时保持能够使用本地数据运行，并保留检索相关的教学流程。这些要求可能彼此冲突：让 notebook 现代化的修复可能意外地降低其可运行性，而保持本地运行的修复又可能删掉过多原有的教学内容。

第三轮会把最新的 notebook 和最终的验证差异交给 Codex。演示的这一部分说明了迭代为什么重要：智能体针对仍然存在的具体问题作出响应，而不是试图预先考虑到所有情况。


In [14]:
iteration_3 = run_iteration(3, iteration_2)
{name: result["validation"]["passed"] for name, result in iteration_3.items()}


{'qdrant_embeddings_search_pre_repair.ipynb': True,
 'getting_started_evals_pre_repair.ipynb': True,
 'knowledge_retrieval_pre_repair.ipynb': True}

## 汇总改进情况

现在我们可以纵览整次运行，而不必手动打开每个中间产物。下面的汇总展示了最重要的信号：哪些产物通过了、还剩多少验证发现，以及是否有差异被带到后续轮次。

对于随附的测试用例，预期的结果形态很简单：一个 notebook 在第 1 次迭代中通过，另一个在第 2 次迭代中通过，难度最高的那个在第 3 次迭代中通过。在真实的维护工作流中，这张表会告诉你循环是在收敛，还是需要更清晰的约束或人工审查。

这份汇总对人工审查也很有用。维护者可以先查看通过/失败的模式，打开仍有差异的记录，只检查那些已可供审查的修复后产物。


In [ ]:
summary = []
for iteration, results in history.items():
    for artifact, record in results.items():
        validation = record["validation"]
        summary.append(
            {
                "iteration": iteration,
                "artifact": artifact,
                "passed": validation["passed"],
                "findings": len(validation["findings"]),
                "remaining_delta": len(validation["remaining_delta"]),
            }
        )

summary


In [16]:
for row in summary:
    print(
        f"iteration={row['iteration']} artifact={row['artifact']} "
        f"passed={row['passed']} findings={row['findings']} delta={row['remaining_delta']}"
    )


iteration=1 artifact=qdrant_embeddings_search_pre_repair.ipynb passed=True findings=0 delta=0
iteration=1 artifact=getting_started_evals_pre_repair.ipynb passed=False findings=0 delta=1
iteration=1 artifact=knowledge_retrieval_pre_repair.ipynb passed=False findings=1 delta=3
iteration=2 artifact=getting_started_evals_pre_repair.ipynb passed=True findings=0 delta=0
iteration=2 artifact=qdrant_embeddings_search_pre_repair.ipynb passed=True findings=0 delta=0
iteration=2 artifact=knowledge_retrieval_pre_repair.ipynb passed=False findings=0 delta=1
iteration=3 artifact=qdrant_embeddings_search_pre_repair.ipynb passed=True findings=0 delta=0
iteration=3 artifact=getting_started_evals_pre_repair.ipynb passed=True findings=0 delta=0
iteration=3 artifact=knowledge_retrieval_pre_repair.ipynb passed=True findings=0 delta=0


## 汇总告诉了我们什么

重要的信号不是 Codex 做了编辑，而是随着循环的运行，剩余的验证差异越来越小。

| 轮次 | 需要关注的信号 | 为什么重要 |
| --- | --- | --- |
| 第 1 次迭代 | 最简单的测试用例通过；更难的测试用例保留少量差异。 | 循环可以先做初步修复，同时把仍需证据的案例带到后续轮次。 |
| 第 2 次迭代 | 中等难度的测试用例在看到验证反馈后通过。 | 运行时反馈和评判反馈变成了有用的修复指令。 |
| 第 3 次迭代 | 难度最高的测试用例通过，或留下一个聚焦的最终差异。 | 循环收敛，或者为人工审查者生成清晰的交接内容。 |

`record.json` 文件让这一切变得可审计。一份有用的记录能回答四个问题：审查发现了什么、Codex 改了什么、notebook 是否执行成功，以及还剩下什么？这正是看起来令人印象深刻的编辑与维护者可以信赖的修复工作流之间的区别。


## 推广为持续循环

上面固定三轮的运行有助于讲解这一模式。而生产环境中的循环应当能够自行决定何时停止。

一个好的循环通常会因以下四种原因之一而停止：验证通过、循环达到最大尝试次数、剩余差异不再变化，或者下一个决策需要人工审查。这些停止条件与修复提示词同样重要。

另一个生产环境中的细节是审计轨迹。为每一轮保留审查发现、修复后的产物、验证结果、验证判断以及剩余差异。这份记录能让维护者理解循环为什么继续、为什么停止，以及哪个产物已可供审查。


In [17]:
def repair_until_done(max_iterations: int = 3) -> dict[int, dict[str, Any]]:
    current_notebooks.update({path.name: path for path in NOTEBOOKS})
    previous = None
    loop_history = {}

    for number in range(1, max_iterations + 1):
        previous = run_iteration(number, previous)
        loop_history[number] = previous
        if all(record["validation"]["passed"] for record in previous.values()):
            break

    return loop_history


## 其他适用场景

notebook 演练只是讲解这一架构的一种方式。只要智能体修改的文件或流程在被接受之前需要的不仅仅是主观审查，同样的模式就能派上用场。

几个高价值的示例：

- **方案优化：** 起草一份更新供专家审查，然后根据剂量规则、时间约束或必需的安全检查对其进行验证。
- **合规整改：** 起草对受监管内容的更新，然后检查必需的措辞、引用、审批以及特定司法辖区的条款是否保持完整。
- **支持知识库更新：** 更新一篇文章，根据当前产品行为或已知解决方案对其进行测试，并将不一致之处带入下一轮。
- **代码现代化：** 替换已弃用的 API，运行测试或静态检查，并用剩余的失败来指导下一次修复。

它们的共同点在于：变更很重要，而且每一轮都需要证据。无论目标是 notebook、策略、方案、支持文章、流水线还是代码库，这个循环都为智能体提供了一种方法，用维护者可以审查的证据来改进它。


## 结论

迭代式修复循环将判断与证明分离开来，因此让智能体驱动的维护工作更易于审查和运营。

审查找出候选问题。修复进行有针对性的编辑。验证执行产物并生成下一轮的差异。当这些阶段之间交换结构化输出时，工作流就更容易检查、重复和调整。

核心思想很简单：不要依赖单次处理，而是让工作流能够从产物中学习、进行有边界的修复，并对真实的验证反馈作出响应。这个小小的改变让智能体驱动的维护工作变得实用得多。